## Generating the Knowledge Distillation Dataset
The following script automates the creation of a highly diverse all_mixed dataset, which is the foundational step for training the lightweight GNN for QoS-aware routing in SDN.

To ensure the student model achieves strong generalization, the training, validation, and test sets must span a wide variety of network scales—ranging from 10 up to 300 nodes. This loop dynamically generates the specific graph topologies, routing path variations, and realistic traffic matrices required to train the robust teacher model (RouteNet-Fermi).

### Dataset Generation
To generate the dataset, we will first need to define the graph topology, routing paths, and traffic matrix for each sample. These parameters will be used by the simulator to calculate the delay, jitter, and drops for each path.

To begin, we will define the graph topology, including the nodes and edges that make up the graph, as well as the scheduling policy and buffer size for each node. We will then create a routing file that defines the paths between the nodes in the topology.

Next, we will generate the traffic matrix, which includes information on the source and destination nodes, average bandwidth, time distribution, packet size and frequency, and ToS for each flow.

Once we have defined these parameters, we can run the simulation and collect performance metrics such as delay, jitter, and drops for each path.

If you need more information on the parameters of the dataset, check out the [input_parameters_glossary.ipynb](input_parameters_glossary.ipynb) notebook, which provides a detailed explanation of each parameter.

In [ ]:
import networkx as nx
import random
import os
import numpy as np

random.seed(42)
np.random.seed(42)

# Define base destination for the generated samples based on your CLI
base_dataset_path = "../data/sim_data_v1"

# We will create specific dataset folders dynamically inside the generation loop,
# but let's ensure the base directory exists.
if not os.path.exists(base_dataset_path):
    os.makedirs(base_dataset_path)
    print(f"Created base directory: {base_dataset_path}")
else:
    print(f"Base path {base_dataset_path} already exists.")

Created base directory: ../data/sim_data_v1


In [ ]:
'''
Generate an undirected graph topology (required by BNNetSimulator parser).
Characteristics:
- 3 ToS levels: 0, 1, 2
- Scheduling policies: Randomly chosen among FIFO, SP, WFQ, DRR.
- Buffer sizes: Randomly chosen from 8000, 16000, 32000, 64000 bits.
'''

# ===============================
# TOPOLOGY GENERATION
# ===============================

def generate_topology(net_size, graph_file):
    G = nx.Graph()
    G.graph["levelsToS"] = 3

    topo_type = random.choice(["barabasi", "erdos", "watts"])

    if topo_type == "barabasi":
        temp_G = nx.barabasi_albert_graph(net_size, random.choice([2, 3]))
    elif topo_type == "erdos":
        temp_G = nx.erdos_renyi_graph(net_size, p=random.uniform(0.05, 0.15))
        max_retries = 10
        retries = 0
        while not nx.is_connected(temp_G):
            if retries >= max_retries:
                # Fall back to Barabási-Albert which is always connected
                temp_G = nx.barabasi_albert_graph(net_size, random.choice([2, 3]))
                print(f"[WARN] Erdős-Rényi failed to connect after {max_retries} tries for net_size={net_size}. Fell back to Barabási-Albert.")
                break
            temp_G = nx.erdos_renyi_graph(net_size, p=random.uniform(0.05, 0.15))
            retries += 1
    else:  # watts
        temp_G = nx.watts_strogatz_graph(net_size, k=4, p=0.3)

    # Nodes
    for n in temp_G.nodes():
        G.add_node(n)

        policy = random.choice(["FIFO", "SP", "WFQ", "DRR"])
        G.nodes[n]["schedulingPolicy"] = policy
        G.nodes[n]["bufferSizes"] = random.choice([8000, 16000, 32000, 64000])

        if policy == "FIFO":
            G.nodes[n]["tosToQoSqueue"] = "0,1,2"
        else:
            G.nodes[n]["tosToQoSqueue"] = "0;1;2"
            if policy in ["WFQ", "DRR"]:
                weights = np.random.dirichlet(np.ones(3)) * 100
                weights = [int(w) for w in weights]
                G.nodes[n]["schedulingWeights"] = f"{weights[0]},{weights[1]},{weights[2]}"

    # Edges
    for u, v in temp_G.edges():
        bw = random.choice([1e4, 5e4, 1e5, 1e6])  # bandwidth
        delay = round(random.uniform(1, 20), 2)   # ms
        G.add_edge(u, v, bandwidth=bw, delay=delay)

    nx.write_gml(G, graph_file)
    return G

In [ ]:
'''
Generate a file with the shortest path routing.
By calling this with randomized weights, we can get N variations of shortest paths.
'''
# ===============================
# ROUTING GENERATION
# ===============================

def generate_routing(G, routing_file):
    routing_type = random.choice(["random", "delay", "bandwidth"])

    for u, v in G.edges():
        if routing_type == "random":
            G[u][v]['weight'] = random.uniform(1, 10)
        elif routing_type == "delay":
            G[u][v]['weight'] = G[u][v]['delay']
        else:  # bandwidth-aware
            G[u][v]['weight'] = 1.0 / G[u][v]['bandwidth']

    with open(routing_file, "w") as f:
        paths = dict(nx.all_pairs_dijkstra_path(G, weight='weight'))

        for src in G.nodes():
            for dst in G.nodes():
                if src == dst:
                    continue
                path = ','.join(map(str, paths[src][dst]))
                f.write(path + "\n")

    # cleanup
    for u, v in G.edges():
        del G[u][v]['weight']

In [ ]:
'''
Generate a traffic matrix file based on all_mixed properties,
but with realistic flow sparsity to prevent O(N^2) explosion.
'''
# ===============================
# TRAFFIC MATRIX GENERATION
# ===============================

def generate_tm(G, max_avg_lbda, traffic_file, sparsity=0.2):

    time_dist_options = ["0", "1", "2,10,5"]

    pkt_dist_profiles = [
        "0,300,0.2,500,0.2,800,0.2,1000,0.2,1500,0.2",
        "0,100,0.1,400,0.3,700,0.2,1200,0.3,1400,0.1",
        "0,200,0.4,600,0.15,900,0.15,1300,0.1,1450,0.2"
    ]

    nodes = list(G.nodes())
    num_hot = max(1, int(0.1 * len(nodes)))
    hot_nodes = random.sample(nodes, num_hot)

    load_level = random.choice(["low", "medium", "high", "congested"])

    load_multiplier = {
        "low": 0.3,
        "medium": 0.6,
        "high": 0.9,
        "congested": 1.2
    }

    with open(traffic_file, "w") as f:
        for src in nodes:
            for dst in nodes:
                if src == dst:
                    continue

                # hotspot-aware sparsity
                prob = sparsity
                if src in hot_nodes or dst in hot_nodes:
                    prob *= 2

                if random.random() > prob:
                    continue

                # heavy-tailed traffic
                avg_bw = int(max_avg_lbda * load_multiplier[load_level] *
                             np.random.pareto(a=2))

                avg_bw = max(100, min(avg_bw, int(max_avg_lbda)))

                td = random.choice(time_dist_options)
                sd = random.choice(pkt_dist_profiles)
                tos = random.choice([0, 1, 2])

                line = f"{src},{dst},{avg_bw},{td},{sd},{tos}"
                f.write(line + "\n")

In [5]:
"""
Generate the data using the structured 'all_mixed' properties.
Creates folders matching: ch22-training-scheduling-topo-<size>-<id>
"""
# Target sizes to generate (expand this array up to 300 based on your compute power)
network_sizes = [10, 50]  # [10, 50, 75, 100, 130, 170] 
topologies_per_size = 1 # Change as needed
routings_per_topology = 1
tms_per_routing = 1

for net_size in network_sizes:
    for topo_id in range(1, topologies_per_size + 1):
        
        # 1. Setup specific dataset directory matching your CLI structure
        dataset_name = f"scheduling-topo-{net_size}-{topo_id}"
        current_dataset_path = os.path.join(base_dataset_path, dataset_name)
        
        graphs_dir = os.path.join(current_dataset_path, "graphs")
        routings_dir = os.path.join(current_dataset_path, "routings")
        tm_dir = os.path.join(current_dataset_path, "tm")
        
        os.makedirs(graphs_dir, exist_ok=True)
        os.makedirs(routings_dir, exist_ok=True)
        os.makedirs(tm_dir, exist_ok=True)
        
        simulation_file = os.path.join(current_dataset_path, "simulation.txt")
        
        with open(simulation_file, "w") as fd:
            # 2. Generate Graph
            graph_filename = f"graph_{net_size}_{topo_id}.txt"
            G = generate_topology(net_size, os.path.join(graphs_dir, graph_filename))
            
            # 3. Generate Routings & TMs
            for r_id in range(routings_per_topology):
                routing_filename = f"routing_{net_size}_{topo_id}_{r_id}.txt"
                generate_routing(G, os.path.join(routings_dir, routing_filename))
                
                for tm_id in range(tms_per_routing):
                    # maxAvgLbda per sample between 400 and 2000 bps
                    sample_max_avg_lbda = random.randint(400, 2000) 
                    
                    tm_filename = f"tm_{net_size}_{topo_id}_{r_id}_{tm_id}.txt"
                    generate_tm(G, sample_max_avg_lbda, os.path.join(tm_dir, tm_filename))
                    
                    # Log simulation line (paths relative to dataset folder for docker)
                    sim_line = f"graphs/{graph_filename},routings/{routing_filename},tm/{tm_filename}\n"
                    fd.write(sim_line.replace("\\", "/"))

print("Dataset generation prep complete. Ready for Docker simulation.")

Dataset generation prep complete. Ready for Docker simulation.


### Automated Sequential Simulation for QoS Metrics
With the varying SDN network scenarios defined, we now use BNNetSimulator to extract the ground-truth QoS metrics (delay, jitter, drops). These exact performance metrics are what RouteNet-Fermi will learn to predict, eventually passing that complex knowledge down to the lightweight student GNN.

Because simulating topologies up to 300 nodes requires significant system memory and compute, the following cells implement a sequential batching process. First, we generate a unique conf.yml file for every scenario. Then, we execute the Docker container using a Python subprocess loop, iterating through each directory one by one to prevent resource exhaustion during the data generation phase.

In [6]:
# Generate the configuration file for EVERY dataset folder
import yaml

# Ensure network_sizes matches the generation block above
network_sizes =  [10, 50]  #  [10, 50, 75, 100, 130, 150, 170, 200, 240, 260, 280, 300] 
topologies_per_size = 1

for net_size in network_sizes:
    for topo_id in range(1, topologies_per_size + 1):
        dataset_name = f"scheduling-topo-{net_size}-{topo_id}"
        current_dataset_path = os.path.join(base_dataset_path, dataset_name)
        
        # Only create a conf.yml if the generation step successfully made the directory
        if os.path.exists(current_dataset_path):
            conf_file = os.path.join(current_dataset_path, "conf.yml")
            
            conf_parameters = {
                "threads": 6, 
                "dataset_name": dataset_name, 
                "samples_per_file": 10, 
                "rm_prev_results": "n", 
                "write_pkt_info": "n", 
            }

            with open(conf_file, 'w') as fd:
                yaml.dump(conf_parameters, fd)
            
print("conf.yml files generated for all existing datasets.")

conf.yml files generated for all existing datasets.


In [7]:
from getpass import getpass
import subprocess
import os

# Unix environments usually require sudo for docker
use_sudo = os.name != 'nt'
pwd = ""

if use_sudo:
    print("Superuser privileges are required to run docker. Introduce sudo password when prompted:")
    pwd = getpass()

print("Starting batch Docker simulations...")

for net_size in network_sizes:
    for topo_id in range(1, topologies_per_size + 1):
        dataset_name = f"scheduling-topo-{net_size}-{topo_id}"
        current_dataset_path = os.path.join(base_dataset_path, dataset_name)
        
        if not os.path.exists(current_dataset_path):
            continue # Skip if directory doesn't exist
            
        # We need the absolute path for Docker volume mounting
        absolute_dataset_path = os.path.abspath(current_dataset_path)
        
        print(f"\n---> Running simulation for {dataset_name}...")
        raw_cmd = f"docker run --rm --mount type=bind,src={absolute_dataset_path},dst=/data bnnupc/bnnetsimulator"
        
        if use_sudo:
            cmd = f"echo {pwd} | sudo -S {raw_cmd}"
        else:
            cmd = raw_cmd
            
        # Run the process sequentially
        try:
            subprocess.run(cmd, shell=True, check=True)
            print(f"Finished {dataset_name}.")
        except subprocess.CalledProcessError as e:
            print(f"Error occurred while simulating {dataset_name}: {e}")

print("\nAll simulations completed!")

Superuser privileges are required to run docker. Introduce sudo password when prompted:
Starting batch Docker simulations...

---> Running simulation for scheduling-topo-10-1...


[sudo] password for abaragithan: INFO:root:0: OK


Finished scheduling-topo-10-1.

---> Running simulation for scheduling-topo-50-1...


[sudo] password for abaragithan: INFO:root:0: OK


Finished scheduling-topo-50-1.

All simulations completed!


### Monitoring Simulation Quality
High-quality, error-free ground truth data is absolutely critical for effective Knowledge Distillation.

Because we are running multiple complex simulations sequentially, the execution cell will only print high-level progress. To verify the detailed status of a specific topology (especially the heavier 200+ node networks), navigate to its respective directory (e.g., ../data/sim_data_v1/scheduling-topo-100-1/) and open the out.log file.

Each out.log contains one line per simulated sample. A status of "Ok" confirms the sample finished properly. Review these logs to guarantee the teacher model will receive stable, accurate network routing data.